# Validate the reporting dataset
Calls the same standard-library validator as the CLI.
If private pipeline config exists, checks its generated CSV; otherwise checks the included snapshot.
Set INPUT_PATH to explicitly choose another CSV and VERIFY_SNAPSHOT to override automatic mode.


In [ ]:
from pathlib import Path
import json
import runpy
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'config/portfolio_contract.json').is_file()), None)
if project_root is None:
    raise RuntimeError('Open the complete repository checkout or Git folder.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Explicit file overrides auto-selection. Plain /Volumes/... paths are supported.
INPUT_PATH = None
VERIFY_SNAPSHOT = None
CONFIG_PATH = project_root / 'config/pipeline.local.json'
snapshot_path = project_root / 'data/processed/portfolio_reporting.csv'
if INPUT_PATH:
    input_path = Path(INPUT_PATH)
elif CONFIG_PATH.is_file():
    from src.data_processing.runtime import load_config
    config = load_config(CONFIG_PATH)
    input_path = Path(config['output_dir']) / 'portfolio_reporting.csv'
else:
    input_path = snapshot_path
if VERIFY_SNAPSHOT is None:
    VERIFY_SNAPSHOT = input_path.resolve() == snapshot_path.resolve()


In [ ]:
module = runpy.run_path(str(project_root / 'src/data_processing/validate_portfolio.py'))
report = module['validate_portfolio'](
    input_path, project_root / 'config/portfolio_contract.json', verify_snapshot=VERIFY_SNAPSHOT
)
print(json.dumps(report, ensure_ascii=False, indent=2))
if report['status'] != 'PASS':
    raise ValueError('Reporting validation failed. Review the issue counts above.')
